# Chapter 4: Building a ReAct Agent

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

## 4.1 Content Types

In [ ]:
from scratch_agent.types import Message, ToolCall, ToolResult, Event

# Message - a text message in the conversation
user_msg = Message(role="user", content="What is 42 * 17?")
print(f"Message: {user_msg}")

# ToolCall - LLM's request to execute a tool
tool_call = ToolCall(
    tool_call_id="call_001",
    name="calculator",
    arguments={"expression": "42 * 17"}
)
print(f"ToolCall: {tool_call}")

# ToolResult - result from tool execution
tool_result = ToolResult(
    tool_call_id="call_001",
    name="calculator",
    status="success",
    content=["714"]
)
print(f"ToolResult: {tool_result}")

# Event - a recorded occurrence during agent execution
event = Event(
    execution_id="exec_001",
    author="user",
    content=[user_msg]
)
print(f"Event: {event}")

## 4.2 ExecutionContext

In [ ]:
from scratch_agent.context import ExecutionContext
from scratch_agent.types import Event, Message

# Create an execution context
ctx = ExecutionContext()
print(f"Execution ID: {ctx.execution_id}")
print(f"Current step: {ctx.current_step}")
print(f"Events: {ctx.events}")

# Add events to the context
user_event = Event(
    execution_id=ctx.execution_id,
    author="user",
    content=[Message(role="user", content="Hello, agent!")]
)
ctx.add_event(user_event)
ctx.increment_step()

print(f"\nAfter adding event:")
print(f"Events count: {len(ctx.events)}")
print(f"Current step: {ctx.current_step}")
print(f"State: {ctx.state}")

## 4.3 Tool Abstraction

In [ ]:
from scratch_agent.tools.base import BaseTool, FunctionTool, tool
from scratch_agent.context import ExecutionContext

# 1. FunctionTool - wrap a plain function
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

calc_tool = FunctionTool(func=calculator)
print(f"FunctionTool name: {calc_tool.name}")
print(f"Tool definition: {calc_tool.tool_definition}")

# Test execution
ctx = ExecutionContext()
result = await calc_tool(ctx, expression="2 ** 10")
print(f"Result: {result}")

# 2. @tool decorator - the simplest way
@tool
def greet(name: str) -> str:
    """Greet a person by name."""
    return f"Hello, {name}!"

print(f"\n@tool decorator name: {greet.name}")
result = await greet(ctx, name="Alice")
print(f"Result: {result}")

# 3. @tool with custom name and description
@tool(name="web_lookup", description="Search the web for information.")
def my_search(query: str) -> str:
    """Internal search implementation."""
    return f"Results for: {query}"

print(f"\nCustom tool name: {my_search.name}")
print(f"Custom description: {my_search.description}")

## 4.4 LLM Communication

In [ ]:
from scratch_agent.llm import LlmClient, LlmRequest, LlmResponse
from scratch_agent.types import Message

# Create an LLM client
llm = LlmClient(model="gpt-4o-mini")
print(f"Model: {llm.model}")

# Build a request
request = LlmRequest(
    instructions=["You are a helpful assistant."],
    contents=[Message(role="user", content="What is 2 + 2?")],
    tools=[],
)
print(f"Request instructions: {request.instructions}")
print(f"Request contents: {request.contents}")

# Generate a response
response = await llm.generate(request)
print(f"\nResponse content: {response.content}")
print(f"Usage: {response.usage_metadata}")

## 4.5 The Agent Class

In [ ]:
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.tools.base import tool
from scratch_agent.tools.search import search_web
from scratch_agent.tools.base import FunctionTool

# Define tools
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

search_tool = FunctionTool(func=search_web)

# Create the agent
agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[calculator, search_tool],
    instructions="You are a helpful research assistant. Use tools when needed.",
    max_steps=5,
    name="research_agent",
)

# Run the agent
result = await agent.run(
    user_input="Search for the current population of Japan and multiply it by 3.",
    verbose=True,
)

print(f"\nFinal output: {result.output}")
print(f"Steps taken: {result.context.current_step}")
print(f"Events count: {len(result.context.events)}")

## 4.6 Structured Output

In [ ]:
from pydantic import BaseModel, Field
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient

class SentimentAnalysis(BaseModel):
    """Structured output for sentiment analysis."""
    sentiment: str = Field(description="The sentiment: positive, negative, or neutral")
    confidence: float = Field(description="Confidence score between 0 and 1")
    reasoning: str = Field(description="Brief explanation of the sentiment classification")

# Create agent with structured output
sentiment_agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[],
    instructions="Analyze the sentiment of the given text.",
    output_type=SentimentAnalysis,
    name="sentiment_agent",
)

result = await sentiment_agent.run(
    user_input="I absolutely loved this movie! The acting was superb and the storyline kept me on the edge of my seat."
)

# The output is a validated SentimentAnalysis object
output = result.output
print(f"Sentiment: {output.sentiment}")
print(f"Confidence: {output.confidence}")
print(f"Reasoning: {output.reasoning}")

## 4.7 GAIA Evaluation with Agent

In [ ]:
from datasets import load_dataset
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.tools.base import FunctionTool
from scratch_agent.tools.search import search_web
from scratch_agent.eval.gaia import is_correct

# Load GAIA benchmark
ds = load_dataset("gaia-benchmark/GAIA", "2023_all", split="validation")
problems = [row for row in ds]
subset = problems[:3]

# Create an agent with tools for GAIA
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

search_tool = FunctionTool(func=search_web)

gaia_agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[calculator, search_tool],
    instructions=(
        "You are solving GAIA benchmark problems. "
        "Use tools as needed to find information and compute answers. "
        "Provide your final answer concisely in the exact format requested."
    ),
    max_steps=8,
    name="gaia_agent",
)

# Evaluate
results = []
for problem in subset:
    agent_result = await gaia_agent.run(user_input=problem["Question"])
    prediction = agent_result.output or ""
    correct = is_correct(str(prediction), problem["Final answer"])
    results.append({
        "task_id": problem["task_id"],
        "correct": correct,
        "prediction": str(prediction)[:100],
        "answer": problem["Final answer"],
    })
    print(f"Task {problem['task_id']}: {'CORRECT' if correct else 'WRONG'}")
    print(f"  Prediction: {str(prediction)[:100]}")
    print(f"  Answer:     {problem['Final answer']}")

correct_count = sum(1 for r in results if r["correct"])
print(f"\nAccuracy: {correct_count}/{len(results)} ({correct_count/len(results)*100:.1f}%)")